# Chapter 3: Forensic Pipeline Evaluation
### Real-world testing of YOLOv5, Roboflow, and Smart Extraction Pipelines
This notebook demonstrates the actual backend logic for the PFE thesis, running inference directly on a test video.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Setup paths
BACKEND_DIR = Path(os.getcwd()).parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
    
from app.services.yolov5_service import YOLOv5Service
from app.services.roboflow_service import RoboflowService
from app.services.vision_rag_service import VisionRAGService

VIDEO_PATH = BACKEND_DIR / "uploads" / "f4603c58-571f-4082-b29a-1b4c67529cc7.mp4"
print(f"Target Video: {VIDEO_PATH.name}")

### 1. Initialize AI Services (YOLOv5, Roboflow, VisionRAG)

In [ ]:
print("Loading YOLOv5s (Local)...")
yolo = YOLOv5Service()

print("Loading Roboflow (Cloud)...")
roboflow = RoboflowService()

print("Loading VisionRAG (Enhancement & Gemini)...")
vision_rag = VisionRAGService()

print("All services loaded!")

### 2. Temporal Sharpness Filter (Finding a Hero Frame)
Extracting Frame 84, which our Laplacian variance filter identified as highly sharp.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 84)
ret, frame_bgr = cap.read()
cap.release()

frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
pil_frame = Image.fromarray(frame_rgb)

plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title("Extracted Hero Frame (Frame 84)")
plt.axis('off')
plt.show()

### 3. Smart Extraction: Multi-Model Inference
Simulating the `smart_extract.py` logic: Running both YOLOv5 and Roboflow on the Hero Frame.

In [ ]:
print("Running YOLOv5 Inference...")
yolo_hits = yolo.detect(pil_frame)
yolo_drawn = yolo.draw_detections(pil_frame, yolo_hits)

print("Running Roboflow Inference (custom-workflow-3)...")
# Send base64 to Roboflow
rf_response = roboflow.detect(pil_frame, workflow_id="custom-workflow-3")
rf_hits = rf_response.get('detections', [])

print(f"\nYOLOv5 detected {len(yolo_hits)} objects:")
for h in yolo_hits:
    print(f"  - {h['class']} (Conf: {h['confidence']:.2f})")
    
print(f"\nRoboflow detected {len(rf_hits)} objects:")
for h in rf_hits:
    print(f"  - {h['class']} (Conf: {h['confidence']:.2f})")

plt.figure(figsize=(10, 6))
plt.imshow(yolo_drawn)
plt.title("YOLOv5 Local Detections")
plt.axis('off')
plt.show()

### 4. Forensic Preprocessing (CLAHE + Upscaling)
Taking the highest confidence Roboflow detection, converting coordinates, applying the 2% safety margin, and enhancing the crop for OCR using CLAHE.

In [ ]:
if rf_hits:
    best_hit = max(rf_hits, key=lambda x: x['confidence'])
    print(f"Selected Target: {best_hit['class']} ({best_hit['confidence']:.2f})")
    
    # 1. Coordinate Conversion (Center to Corner)
    x = int(best_hit['x'] - best_hit['width']/2)
    y = int(best_hit['y'] - best_hit['height']/2)
    w = int(best_hit['width'])
    h = int(best_hit['height'])
    
    # 2. Safety Margin (2%)
    mx, my = int(w * 0.02), int(h * 0.02)
    x1, y1 = max(0, x + mx), max(0, y + my)
    x2, y2 = min(frame_rgb.shape[1], x + w - mx), min(frame_rgb.shape[0], y + h - my)
    
    # 3. Raw Crop
    raw_crop = frame_rgb[y1:y2, x1:x2]
    
    # 4. Enhance using VisionRAGService (CLAHE + Unsharp Mask)
    # VisionRAGService expects BGR, so we pass BGR
    raw_crop_bgr = frame_bgr[y1:y2, x1:x2]
    enhanced_crop_bgr = vision_rag.enhance_crop_for_ocr(raw_crop_bgr)
    enhanced_crop_rgb = cv2.cvtColor(enhanced_crop_bgr, cv2.COLOR_BGR2RGB)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    axes[0].imshow(raw_crop)
    axes[0].set_title("Raw Bounding Box Crop (w/ 2% Margin)")
    axes[0].axis('off')
    
    axes[1].imshow(enhanced_crop_rgb)
    axes[1].set_title("Forensically Enhanced Crop (CLAHE + Lanczos4)")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No Roboflow detections found to crop.")